# Building Trusted GeoAI

This notebook continues from `SpatialGrounding.ipynb`: it takes the ranked
recommendations generated there, runs them through a human-in-the-loop
review step with a durable audit trail (`traffic_safety.governance`), and
demonstrates swapping the recommendation pipeline to a local, sovereign
model (`OllamaLLMClient`) with no other code changes.

In [8]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [9]:
import os
from IPython.display import display, Markdown

from arcgis.gis import GIS

from traffic_safety.recommendations import get_risk_layer, load_context_geometries, rank_recommendations
from traffic_safety.governance import (
    OllamaLLMClient,
    ReviewDecision,
    load_audit_trail,
    request_human_review,
)

In [10]:
home_portal = GIS("home")
risk_layer_item_id = os.getenv("RISK_LAYER_ITEM_ID")
risk_layer = get_risk_layer(home_portal, risk_layer_item_id)
audit_log_path = os.getenv("AUDIT_LOG_PATH", "audit_trail.jsonl")

## Recap: spatial context from Session 2

For anyone joining fresh, Session 2 (`SpatialGrounding.ipynb`) built evidence
not just from the risk score, but from nearby real-world context (schools
within a 200m walking buffer of each top-risk location) and persisted it to
disk. This session reloads that same context layer and per-location search
geometries rather than recomputing them, so the evidence package handed to
the local model below is identical to Session 2's - only the generation
mechanism changes.

In [11]:
def get_context_layer(portal: GIS, item_id: str):
    """Retrieves a hosted feature layer by item id (here, the schools context layer)."""
    item = portal.content.get(item_id)
    return item.layers[0]

schools_layer_item_id = os.getenv("SCHOOLS_LAYER_ITEM_ID")
context_layer = get_context_layer(home_portal, schools_layer_item_id)

output_dir = os.getenv("OUTPUT_DIRECTORY")
context_geometries_path = os.path.join(output_dir, "context_geometries_buffer.geojson")
context_geometries = load_context_geometries(context_geometries_path)

## Local, sovereign inference

`OllamaLLMClient` implements the exact same `generate(prompt)` interface used
by `TemplateLLMClient` in the previous session. Requires a local Ollama
server (`ollama serve`) with the configured model pulled, and the optional
`local-inference` dependency group (`uv sync --extra local-inference`).

In [12]:
local_llm_client = OllamaLLMClient(model=os.getenv("OLLAMA_MODEL", "llama3.2"))
recommendations = rank_recommendations(
    risk_layer,
    context_layer=context_layer,
    context_geometries=context_geometries,
    context_method="buffer",
    top_n=5,
    llm_client=local_llm_client,
)

## Human-in-the-loop review

Every recommendation is reviewed before any downstream action is taken. The
reviewer sees the full evidence package (risk score, confidence range, years
observed, supporting context) and records approve / reject / needs-review.
`request_human_review` persists the decision as an append-only audit
record.

In [13]:
reviewer = os.getenv("REVIEWER_NAME", "demo-reviewer")

for recommendation in recommendations:
    display(Markdown(recommendation["recommendation"]))
    answer = input("Approve this recommendation? [y/n/skip]: ").strip().lower()
    decision = {
        "y": ReviewDecision.APPROVED,
        "n": ReviewDecision.REJECTED,
    }.get(answer, ReviewDecision.NEEDS_REVIEW)

    request_human_review(recommendation, decision=decision, reviewer=reviewer, audit_log_path=audit_log_path)

Based on the information provided, I would recommend prioritizing Location 2 for traffic safety review. Here's why:

1. Persistent risk score: A persistent risk score of 3.00 indicates that Location 2 has consistently been identified as a location of concern for traffic safety. This suggests that there is a recurring issue that needs to be addressed.
2. Range [0.0, 3.0]: The fact that the risk score is within the range of 0.0 to 3.0 indicates that the issue is moderate to severe. A score of 3.0 is not trivial, and it's essential to investigate further.
3. Observed across 7 year(s): The fact that the risk score has been persistent for 7 years suggests that the issue has been ongoing and has not been addressed. This indicates that there may be a systemic or underlying problem that needs to be addressed.
4. 0 relevant context feature(s) nearby: The lack of relevant context features nearby means that the issue at Location 2 may be isolated or unique. This makes it essential to thoroughly investigate the location to understand the root cause of the problem.

Overall, the combination of a persistent risk score, moderate to severe risk level, ongoing issue, and lack of context suggests that Location 2 should be prioritized for traffic safety review.

Approve this recommendation? [y/n/skip]:  skip


Based on the information provided, I would recommend prioritizing Location 5 for traffic safety review due to the following reasons:

1. **Persistent risk score**: The persistent risk score of 3.00 indicates that there is a significant risk associated with this location, which warrants further investigation.
2. **Long observation period**: The location has been observed for 7 years, suggesting that the issue is likely to be a persistent problem rather than a one-time event.
3. **Relevant context feature(s) nearby**: The presence of relevant context features nearby may indicate that the location is part of a larger safety issue or that the problem is not isolated to this location alone.

Overall, while a risk score of 3.00 may not seem alarming at first, the combination of a persistent risk score, long observation period, and relevant context features nearby suggests that Location 5 is a priority for traffic safety review.

Approve this recommendation? [y/n/skip]:  skip


**Location 6: Risk Score 3.00**

**Recommendation:** Prioritize Location 6 for traffic safety review due to its high risk score of 3.00.

**Risk Factors:**

1. **Persistent Risk Score:** The risk score has been consistently high over the past 7 years, indicating a potential ongoing issue with traffic safety.
2. **No Relevant Context Features:** The lack of nearby context features (e.g., intersections, pedestrian zones, school zones) suggests that the risk score may be related to specific factors at this location, making it a priority for review.

**Traffic Safety Review Recommendations:**

1. Conduct a thorough risk assessment to identify the root causes of the persistent risk score.
2. Review traffic data and patterns to determine if there are any correlations between the risk score and specific times of day, day of the week, or weather conditions.
3. Investigate potential contributing factors, such as:
	* Speed or volume of traffic
	* Pedestrian or cyclist activity
	* Road geometry or design
	* Lighting or visibility issues
4. Consider implementing safety measures to mitigate the risk, such as:
	* Speed limit reductions
	* Traffic calming measures (e.g., speed humps, chicanes)
	* Improved lighting or signage
	* Enhanced pedestrian or cyclist infrastructure

By prioritizing Location 6 for traffic safety review, we can identify and address potential safety concerns, reducing the risk of accidents and improving the overall safety of the area.

Approve this recommendation? [y/n/skip]:  skip


**Recommendation:**

Location 7 should be prioritized for a traffic safety review due to its persistent risk score of 3.00, which indicates a moderate to high risk of accidents. The fact that the risk score is observed across 7 years suggests that this location has been consistently hazardous, which warrants further investigation and potential mitigation measures.

**Key Factors Supporting the Recommendation:**

1. **Persistent risk score**: A risk score of 3.00 is above the recommended threshold, indicating a moderate to high risk of accidents.
2. **Duration**: The risk score has been observed across 7 years, suggesting that this location has been consistently hazardous.
3. **Relevant context feature(s)**: Although the report only mentions "1 relevant context feature(s)" nearby, the fact that there is at least one relevant feature suggests that there may be additional factors contributing to the risk score.
4. **Lack of improvement**: The persistent risk score suggests that the location has not shown any improvement over time, which may indicate a need for more targeted interventions.

**Next Steps:**

1. Conduct a thorough traffic safety review of Location 7, including analysis of historical accident data, traffic volume, and other relevant factors.
2. Identify potential mitigation measures, such as traffic calming interventions, road surface improvements, or improved lighting.
3. Develop a plan to implement these measures and monitor their effectiveness in reducing the risk score.
4. Consider engaging with local stakeholders, such as community members, businesses, and emergency services, to gather feedback and support for the proposed interventions.

By prioritizing Location 7 for a traffic safety review, we can identify potential hazards and implement targeted interventions to reduce the risk of accidents and improve traffic safety.

Approve this recommendation? [y/n/skip]:  skip


Based on the provided information, here's a recommendation for prioritizing Location 8 for traffic safety review:

**Location 8: Priority for Traffic Safety Review**

**Risk Score:** 3.00 (range [0.0, 3.0])
**Duration:** 7 years
**Context Features:** 1 relevant context feature nearby

**Recommendation:**

Given the persistent risk score of 3.00, which falls within the moderate to high-risk category, Location 8 warrants a thorough traffic safety review. The duration of 7 years suggests that this location has been consistently identifying as a safety concern, indicating a potential need for proactive measures to mitigate risks.

**Prioritization Rationale:**

1. **Persistent risk score:** A risk score of 3.00 is considered moderate to high-risk, indicating a potential safety concern that requires attention.
2. **Duration:** The 7-year duration suggests that this location has been consistently identified as a safety concern, warranting a thorough review.
3. **Context features:** The presence of 1 relevant context feature nearby may be contributing to the persistent risk score, and identifying and addressing this feature could help mitigate the risk.

**Recommendations:**

1. Conduct a thorough traffic safety review of Location 8, including an assessment of the risk score, context features, and potential safety concerns.
2. Identify and address the 1 relevant context feature nearby, if possible, to mitigate the risk score.
3. Consider implementing safety measures or countermeasures to reduce the risk score and ensure the safety of users.
4. Monitor the location's risk score and adjust safety measures as needed.

By prioritizing Location 8 for a traffic safety review, you can identify and address potential safety concerns, reducing the risk of accidents and ensuring a safer environment for users.

Approve this recommendation? [y/n/skip]:  skip


## Audit trail

Every review decision is recorded alongside the full recommendation and
evidence that produced it, giving a transparent, reproducible record of who
approved what, and why.

In [7]:
load_audit_trail(audit_log_path)

[{'recommendation': {'location_id': 2,
   'risk_score': 3.0,
   'years_observed': 7,
   'risk_range': [0.0, 3.0],
   'context_feature_count': 0,
   'recommendation': "Based on the provided information, I would agree that Location 2 should be prioritized for a traffic safety review. Here's why:\n\n1. **Persistent risk score**: A risk score of 3.00, which is at the upper end of the range, indicates a moderate to high risk of accidents or safety issues. The fact that this score has been observed across 7 years suggests that the risk is not a one-time event, but rather a persistent concern.\n2. **Relevance of context features**: Although there are no relevant context features nearby, the location itself is still considered at risk. This implies that there may be underlying factors contributing to the risk, such as design or operational issues.\n3. **Lack of mitigating factors**: With 0 relevant context features nearby, there is little to suggest that external factors are mitigating the ris